# Can you recover a planet from public velocities? Fit iota Dra b's eccentric orbit

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/gaia_03_inner_planet_from_velocities.ipynb)

The lab's paper on iota Draconis fit two decades of velocities and found two companions: iota Dra b, on an eccentric 511-day orbit, and iota Dra c, much further out ([Hill et al. 2021](https://arxiv.org/abs/2107.13583)). In this notebook you build orbitize!'s input from the public velocities, fit both companions to the velocities alone, and check b against the paper. This is the starting point for the joint fit with Hipparcos and Gaia.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# orbitize! is built from source in Colab, so this takes about two minutes the first time.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("orbitize") is None:
    %pip install -q "habtrain[orbits] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/gaia_03_inner_planet_from_velocities.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## The velocities

Two public tables hold most of the paper's velocities:
- **Lick Observatory, 2000 to 2005:** 119 velocities ([Butler et al. 2006](https://arxiv.org/abs/astro-ph/0607493), VizieR J/ApJ/646/505)
- **The Automated Planet Finder (APF), 2018 to 2021:** 456 velocities (Hill et al. 2021, VizieR J/AJ/162/211)

The paper also used 46 more Lick velocities, taken between these two sets, that are not in a public table, so your fit will not match the paper exactly. orbitize! wants velocities in km/s and dates as MJD (Julian date minus 2,400,000.5), with a label for each telescope, so the cell below converts both tables.

In [ ]:
# Every public velocity of iota Dra, written to iota_dra_rv.csv in the form orbitize! reads.
rv = ht.orbits.iota_dra_velocities()
print(rv.instrument.value_counts().to_string())

# The velocities over time, one colour per telescope.
fig_rv, ax = plt.subplots(figsize=(10, 4), dpi=110)
for inst, colour in (("lick", "#3b6ea5"), ("apf", "#d1495b")):
    d = rv[rv.instrument == inst]
    ax.errorbar(d.epoch, d.rv * 1000, yerr=d.rv_err * 1000, fmt="o", ms=3, color=colour, label=inst.upper())
ax.set_xlabel("Date (MJD)"); ax.set_ylabel("Velocity (m/s)"); ax.legend()
ax.set_title("iota Dra: every public velocity", fontsize=11)
fig_rv.text(0.01, 0.005, "Data: Butler et al. 2006 (Lick); Hill et al. 2021 (APF). VizieR J/ApJ/646/505, J/AJ/162/211.", fontsize=7, color="#666")
plt.show()

Look for the gap of more than twelve years between the two telescopes, and the sharp swings: b's orbit is so stretched that the star's velocity changes fast near closest approach and slowly the rest of the time.

## An eccentric orbit

The bootcamp fitted circles. An eccentric orbit needs four numbers for its shape and timing, besides the size of the swing:
- **Period:** how long one orbit takes.
- **Eccentricity (e):** how stretched it is, from 0 for a circle towards 1.
- **Argument of periastron (ω):** where in the orbit the closest approach (periastron) happens, as an angle.
- **Time of periastron:** when the companion passes closest. orbitize! stores it as a fraction of the period, called tau.

orbitize! describes each orbit by its **semi-major axis** (half the long axis, in AU) instead of its period, and fits the companion's mass directly. With the orbit held edge on (inclination 90 degrees), the mass it fits from velocities is the minimum mass.

Two traps:
- Velocity papers quote the **star's** argument of periastron. orbitize! uses the **companion's**, which is the star's plus 180 degrees (π radians).
- orbitize! works in radians, km/s and MJD.

## Why fit in two stages

A fit with 20 numbers free, started anywhere, can wander off: in testing, a free search found a negative mass and, another time, an eccentricity of 0.58 for b. So fit in two stages:
1. Hold b's shape at the paper's values and find c and the two telescopes' offsets, trying eight different starting guesses for c and keeping the best.
2. Start from that answer, free b's shape and the two jitters, and fit everything together.

Each stage finds the single most likely orbit (a best fit), not a posterior. The joint fit in the next step starts its walkers in a small ball around this answer. Together the stages take about three minutes.

In [ ]:
# The system: iota Dra with two companions (b first, then c), velocities only.
s = ht.orbits.iota_dra_system()
idx = s.param_idx
mjup = ht.orbits.MJUP_IN_MSUN

# Start: b at the paper's orbit, edge on; c unknown.
start = ht.orbits.b_start(s)

# Stage 1: b's shape held. Free: the two offsets (gamma), c's orbit and mass, and b's timing.
free1 = ["gamma_apf", "gamma_lick", "sma2", "ecc2", "aop2", "tau2", "m2", "tau1", "aop1"]
rng = np.random.default_rng(2)
best = (-np.inf, None)
for attempt in range(8):
    # A random guess for c: 8 to 40 AU, eccentricity 0.1 to 0.7, 5 to 40 Jupiter masses.
    guess = [0, 0, rng.uniform(8, 40), rng.uniform(0.1, 0.7), rng.uniform(0, 2 * np.pi), rng.uniform(0, 1),
             rng.uniform(5, 40) * mjup, start[idx["tau1"]], start[idx["aop1"]]]
    L, q = ht.orbits.best_fit(s, start, free1, guess)
    if L > best[0]:
        best = (L, q)
stage1 = best[1]
print("Stage 1, b held:", ht.orbits.describe(s, stage1))
print(f"c: {stage1[idx['sma2']]:.1f} AU, minimum mass {stage1[idx['m2']] / mjup:.1f} Jupiter masses")

In [ ]:
# Stage 2: start from stage 1 and free b's size, shape and minimum mass, and both jitters.
free2 = free1 + ["ecc1", "m1", "sma1", "sigma_apf", "sigma_lick"]
L2, stage2 = ht.orbits.best_fit(s, stage1, free2, [stage1[idx[k]] for k in free2])
mine = ht.orbits.describe(s, stage2)
print("Stage 2, everything free:", mine)

# Sanity check: b must come back near eccentricity 0.70.
if abs(mine["eccentricity"] - 0.70) > 0.05:
    print("b's eccentricity has moved far from 0.70: the fit has found a different orbit. Run stage 1 again with more attempts.")

## Check against the paper

The table below puts your stage 2 numbers next to the paper's table 3, which fit the same kind of model with RadVel, a different code. Your numbers are a best fit with no error bars, so compare them with the paper's range: how many of the paper's sigmas away is each one?

In [ ]:
# Your numbers against the paper's, and how many of the paper's sigmas away (yours has no error bar of its own).
rows = []
for name, (value, minus, plus) in ht.orbits.IOTA_DRA_B_PUBLISHED.items():
    yours = mine[name]
    sigma = (yours - value) / (plus if yours > value else minus)
    rows.append({"": name, "yours": yours, "paper": f"{value} (-{minus}, +{plus})", "paper's sigmas away": round(sigma, 1)})
import pandas as pd
comparison = pd.DataFrame(rows)
comparison

> **Write down: how your fit compares**
>
> - Which of b's numbers land within 1 sigma of the paper, and which do not? One line. Put it in `MATCH`.
> - Why might the Lick jitter differ more than the rest? One line. Put it in `LICK_DIFFERS`.
> - Why is c barely pinned down by these velocities? Look at its period against the time the data covers. One or two lines. Put it in `WHY_C_IS_LOOSE`.
>
> **Where to look:** the comparison table, the velocity plot, and the note on the missing Lick velocities.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

c's orbit is many decades long. Compare that with the span from the first Lick velocity to the last APF one.

</details>

In [ ]:
MATCH = ...            # one line, in quotes
LICK_DIFFERS = ...     # one line, in quotes
WHY_C_IS_LOOSE = ...   # one or two lines, in quotes

# Keep the answers, with your fitted numbers and the comparison, for your hand-in table.
ht.record.save("inner_planet", mine, "iota Dra b and the jitters, from your stage 2 fit")
ht.record.save("comparison", comparison, "your fit against Hill et al. 2021, table 3")
ht.record.save("match", MATCH, "which numbers match the paper")
ht.record.save("lick_differs", LICK_DIFFERS, "why the Lick jitter may differ")
ht.record.save("why_c_is_loose", WHY_C_IS_LOOSE, "why c is barely pinned down by velocities")

### Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `inner-planet-fitted`, and download them.

In [ ]:
# Your answers and the velocity plot, saved as your hand-in.
ht.record.handin("inner-planet-fitted", ["inner_planet", "comparison", "match", "lick_differs", "why_c_is_loose"],
                 figures={"velocities": fig_rv})